# ArcGIS Online item "Terms of Use" editor

**Welcome!**  
This notebook helps you scan, review, and update multiple ArcGIS Online items at scale. It targets the "Terms of Use" section (the JSON `licenseInfo` field), searching for specific strings within the existing HTML and replacing them with modifications of your choice. It could be modified to selectively target other sections of the Item's "Description" page. It was developed to find logo images that no longer resolve and replace them with viable ones.

To keep the interface clean, most of the functions are located in helper_functions.py located in /arcgis/home.

**Where this notebook can run**  
- ArcGIS Online Notebook (JupyterLab-style).
- VS Code on macOS (local Jupyter kernel).
- VS Code on Windows (local Jupyter kernel).

**How to use this notebook**  
- Start with **1. Setup and authenticate** and run the code cell to connect to your organization.
- Run **2. Scan your content** to find matching terms.
- Save scan outputs, optionally run a secondary scan for additional terms, and review strict matches.
- Do a **dry run** to see exactly what would change. 
- Use the dry run output to generate a webpage report you can use to visually compare old and new HTML blocks.
- Commit updates only after validating the dry-run/report outputs.

**Notes**  
- Some org-wide scans take time, especially for large orgs; progress messages are printed as individual users traversed.
- The workflow is designed to be safe-by-default: review first, then update.
- If you need to restart, use Kernel restart and rerun from setup.
- When running in ArcGIS Online, you can select View > Collapse All Code in the toolbar above to hide the code for a cleaner view.

**tldr;**

In [ ]:
# Run this cell to display Notebook details
from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does**  
- Authenticates to ArcGIS Online
- Scans an entire Organization's Item Description page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames:`scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user friendly side-by-side HTML review report for visual QA.
- Applies updates and edits the itemsonly after explicit confirmation, then exports success/error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))

## 1. Setup and authenticate

In [ ]:
# When running in ArcGIS Online, you can select View > Collapse All Code in the toolbar above to hide the code for a cleaner view.
print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

# Support local VS Code (macOS/Windows) and ArcGIS Online Notebook locations for helper_functions.py
NOTEBOOK_DIR = Path.cwd()
CANDIDATE_HELPER_DIRS = [NOTEBOOK_DIR, Path("/arcgis/home")]
for p in CANDIDATE_HELPER_DIRS:
    helper_file = p / "helper_functions.py"
    if helper_file.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from helper_functions import (
    detect_environment,
    default_output_dir_str,
    default_output_path_str,
    initialize_ui,
    set_runtime_context,
    setup_notebook_btn,
    run_primary_scan_btn,
    save_scan_outputs_btn,
    run_secondary_scan_btn,
    run_strict_match_filter_btn,
    dry_run_btn,
    create_report_btn,
    export_dry_run_btn,
    apply_updates_btn,
    export_final_results_btn,
    OFFICIAL_TOU_HTML_FILE,
 )

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {
    "gis": None,
    "matches_df": None,
    "errors_df": None,
    "all_items_df": None,
    "TARGET_STRINGS": [],
    "output_dir": default_output_dir_str(),
    "official_tou_html_file": OFFICIAL_TOU_HTML_FILE,
}
set_runtime_context(context)

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")
print(f"Default output folder: {context['output_dir']}")

output1 = initialize_ui("output")
context["output1"] = output1

# Create widgets
btn1 = initialize_ui("button", description="Setup Notebook", width="200px")
display(btn1)
btn1.on_click(setup_notebook_btn)
display(output1)

## 2. Scan your content
This cell scans your content for the text strings and/or HTML you input into the box.

In [ ]:
# Cell 2: Define terms and scan org content for licenseInfo matches
output2 = initialize_ui("output")
context["output2"] = output2

help2 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "<strong>Enter text or HTML to find in the Terms of Use section.</strong> "
        "Use CSV-style input (comma-separated).<br>"
        "Wrap text with spaces in double quotes, for example: "
        "&quot;&lt;a href=https://example.com&gt;link&lt;/a&gt;&quot;."
        "</div>"
    )
)

input2 = initialize_ui(
    "textarea",
    value='https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png, esrilogo',
    description="",
    width="700px",
    height="70px",
)
context["input2"] = input2
btn2 = initialize_ui("button", description="Scan for items", width="200px")
display(widgets.VBox([help2, input2, btn2, output2]))

btn2.on_click(run_primary_scan_btn)

## 3. Save the scan results to csv files
This cell saves the scan. You can change the filenames and location if you'd like

In [ ]:
# Cell 3: Save latest scan outputs to CSV
output3 = initialize_ui("output")
context["output3"] = output3
input3_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="700px",
)
context["input3_matches"] = input3_matches
input3_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["input3_errors"] = input3_errors
input3_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="700px",
)
context["input3_all_items"] = input3_all_items
btn3 = initialize_ui("button", description="Save files")
display(widgets.VBox([input3_matches, input3_errors, input3_all_items, btn3, output3]))

btn3.on_click(save_scan_outputs_btn)

## 4. Optionally reload results from a previous scan
use csv

In [ ]:
# Cell 4: Optionally load results from a previous scan
output4 = initialize_ui("output")
context["output4"] = output4

input4_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="900px",
)
input4_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="900px",
)
input4_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="900px",
)
btn4 = initialize_ui("button", description="Load previous scan files", width="240px")

def load_previous_scan_btn(_):
    with output4:
        output4.clear_output()

        matches_path = (input4_matches.value or "").strip()
        errors_path = (input4_errors.value or "").strip()
        all_items_path = (input4_all_items.value or "").strip()

        if not matches_path or not Path(matches_path).exists():
            print(f"Matches file not found: {matches_path}")
            return
        if not all_items_path or not Path(all_items_path).exists():
            print(f"All-items file not found: {all_items_path}")
            return

        context["matches_df"] = pd.read_csv(matches_path, dtype={"item_id": str})

        if errors_path and Path(errors_path).exists():
            try:
                context["errors_df"] = pd.read_csv(errors_path)
            except pd.errors.EmptyDataError:
                context["errors_df"] = pd.DataFrame(columns=["username", "error"])
        else:
            context["errors_df"] = pd.DataFrame(columns=["username", "error"])
            print(f"Errors file not found or blank, using empty table: {errors_path}")

        context["all_items_df"] = pd.read_csv(all_items_path, dtype={"item_id": str})

        print(
            f"Reloaded: matches={len(context['matches_df'])}, "
            f"errors={len(context['errors_df'])}, "
            f"all_items={len(context['all_items_df'])}"
        )

display(widgets.VBox([input4_matches, input4_errors, input4_all_items, btn4, output4]))
btn4.on_click(load_previous_scan_btn)

## 5. Secondary scan
This cell saves you time if you want to search additional terms.
If you want to search again, click the SECONDARY_SCAN check box and add the new terms to NEW_TERMS and run the cell below

In [ ]:
# Cell 5: Optional secondary scan using new terms and excluding previous matches
output5 = initialize_ui("output")
context["output5"] = output5
checkbox5 = initialize_ui("checkbox", description="Run secondary scan with new search terms?", value=False)
context["checkbox5"] = checkbox5
help5 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "As above, use CSV-style input separated by commas.<br>"
        "Wrap text with spaces in double quotes, for example: &quot;text with spaces&quot;."
        "</div>"
    )
)
input5 = initialize_ui(
    "textarea",
    value='https://www.esri.com/content/dam/esrisites/en-us/common/logos/esri-logo.jpg',
    description="",
    width="700px",
    height="70px",
)
context["input5"] = input5

btn5 = initialize_ui("button", description="Run secondary scan")
display(widgets.VBox([checkbox5, help5, input5, btn5, output5]))

btn5.on_click(run_secondary_scan_btn)

## 6. Optionally narrow your original query

In [ ]:
# Cell 6: Optionally filter the scan result to rows containing the exact text below
output6 = initialize_ui("output")
context["output6"] = output6
input6 = initialize_ui(
    "text",
    value="https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png",
    description="Exact text:",
    width="700px",
)
context["input6"] = input6
btn6 = initialize_ui("button", description="Filter exact matches", width="200px")
display(widgets.VBox([input6, btn6, output6]))

btn6.on_click(run_strict_match_filter_btn)

## 7. Dry run

In [ ]:
# Cell 7: Do a dry-run before making any changes. Modify the input file to use your own custom HTML.
output7 = initialize_ui("output")
context["output7"] = output7
input7 = initialize_ui(
    "text",
    value=context.get("official_tou_html_file", OFFICIAL_TOU_HTML_FILE),
    description="Input HTML file:",
    width="900px",
)
btn7 = initialize_ui("button", description="Build dry run", width="180px")

def run_dry_run_with_file(_):
    entered = (input7.value or "").strip()
    context["official_tou_html_file"] = entered or OFFICIAL_TOU_HTML_FILE
    dry_run_btn(_)

display(widgets.VBox([input7, btn7, output7]))

btn7.on_click(run_dry_run_with_file)

## 8. Export dry run results

In [ ]:
# Cell 8: Export the dry-run plan for record-keeping and review
output8 = initialize_ui("output")
context["output8"] = output8
input8_csv_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_results.csv"),
    description="Output CSV:",
    width="700px",
)
context["input8_csv_name"] = input8_csv_name
btn8 = initialize_ui("button", description="Export dry-run CSV", width="200px")
display(widgets.VBox([input8_csv_name, btn8, output8]))

btn8.on_click(export_dry_run_btn)

## 9. Create report

In [ ]:
# Cell 9: Generate an HTML report for review before updating items
output9 = initialize_ui("output")
context["output9"] = output9
input9_report_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_report.html"),
    description="Report HTML:",
    width="700px",
)
context["input9_report_name"] = input9_report_name
input9_selection_json = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="Selected IDs download:",
    width="700px",
)
context["input9_selection_json"] = input9_selection_json
btn9 = initialize_ui("button", description="Create report", width="200px")
display(widgets.VBox([input9_report_name, input9_selection_json, btn9, output9]))

btn9.on_click(create_report_btn)

## 10. Commit updates

In [ ]:
# Cell 10: Apply updates only after reviewing the dry run report 
output10 = initialize_ui("output")
context["output10"] = output10
input10_ids = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="JSON IDs file:",
    width="700px",
)
context["input10_ids"] = input10_ids

input10_confirm = initialize_ui(
    "text",
    value="",
    description="Type APPLY UPDATES to confirm:",
    width="700px",
)
context["input10_confirm"] = input10_confirm

btn10 = initialize_ui("button", description="Execute update", width="180px")
display(widgets.VBox([input10_ids, input10_confirm, btn10, output10]))

btn10.on_click(apply_updates_btn)

## 11. Export final results

In [ ]:
# Cell 11: Export final update results to CSV files for record-keeping
output11 = initialize_ui("output")
context["output11"] = output11
input11_success_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_successes.csv"),
    description="Success CSV:",
    width="700px",
)
context["input11_success_csv"] = input11_success_csv
input11_errors_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["input11_errors_csv"] = input11_errors_csv
btn11 = initialize_ui("button", description="Export final CSVs", width="180px")
display(widgets.VBox([input11_success_csv, input11_errors_csv, btn11, output11]))

btn11.on_click(export_final_results_btn)